In [1]:
"""
Ablation_Advection.py

Ablation comparing:
1) KAPI advection solver trained over a parametric family
2) Single-instance shallow PINN obtained by removing the meta-network

Fixed ablation task:
    (x0, nu) = (0.50, 0.07)

Notebook-safe: uses parse_known_args().

Outputs:
- ablation_summary.csv
- ablation_summary.json
- ablation_outputs.npz
- ablation_training_curves.png
- ablation_comparison.png
- ablation_geometry.png
"""

from __future__ import annotations

import csv
import json
import time
import math
import random
import argparse
from pathlib import Path
from dataclasses import dataclass
from typing import Dict

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ============================================================
# Fixed ablation task
# ============================================================
X0_FIXED = 0.50
NU_FIXED = 0.07

# Meta-training ranges
X0_MIN, X0_MAX = 0.20, 0.80
NU_MIN, NU_MAX = 0.03, 0.12
NU_EASY = 0.06

# ============================================================
# Utilities
# ============================================================
def get_device(use_gpu_flag: bool) -> torch.device:
    if use_gpu_flag and torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def periodic_distance_torch(x: torch.Tensor, mu: torch.Tensor) -> torch.Tensor:
    return torch.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_distance_numpy(x: np.ndarray, mu: np.ndarray) -> np.ndarray:
    return np.remainder(x - mu + 0.5, 1.0) - 0.5


def periodic_gaussian_torch(x: torch.Tensor, x0: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    d = periodic_distance_torch(x, x0)
    return torch.exp(-(d ** 2) / (2.0 * nu ** 2))


def periodic_gaussian_numpy(x: np.ndarray, x0: float, nu: float) -> np.ndarray:
    d = periodic_distance_numpy(x, x0)
    return np.exp(-(d ** 2) / (2.0 * nu ** 2))


def fourier_features(t: torch.Tensor, n_freq: int) -> torch.Tensor:
    feats = [t]
    for k in range(1, n_freq + 1):
        feats.append(torch.sin(2.0 * math.pi * k * t))
        feats.append(torch.cos(2.0 * math.pi * k * t))
    return torch.cat(feats, dim=-1)


# ============================================================
# Meta-KAPI / meta-SPINN model
# ============================================================
class DynamicMetaSPINN(nn.Module):
    def __init__(
        self,
        M: int = 32,
        hidden_task: int = 64,
        hidden_time: int = 96,
        n_fourier: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.25,
        vel_scale: float = 0.25,
    ) -> None:
        super().__init__()
        self.M = M
        self.n_fourier = n_fourier
        self.h_min = h_min
        self.h_max = h_max
        self.vel_scale = vel_scale

        self.task_net = nn.Sequential(
            nn.Linear(2, hidden_task),
            nn.Tanh(),
            nn.Linear(hidden_task, hidden_task),
            nn.Tanh(),
        )
        self.init_head = nn.Linear(hidden_task, 4 * M)

        in_dyn = hidden_task + (1 + 2 * n_fourier)
        self.dyn_net = nn.Sequential(
            nn.Linear(in_dyn, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, 4 * M),
        )

    def normalize_p(self, p: torch.Tensor) -> torch.Tensor:
        mean = p.new_tensor([0.5, 0.075])
        std = p.new_tensor([0.3, 0.03])
        return (p - mean) / std

    def basis_params(self, p: torch.Tensor, t: torch.Tensor):
        """
        p: (B,2)
        t: (B,N,1)
        returns alpha, xi, h, gate with shapes (B,N,M)
        """
        B, N, _ = t.shape
        p_norm = self.normalize_p(p)
        emb = self.task_net(p_norm)

        init_out = self.init_head(emb).view(B, 4, self.M)
        gate_logits = init_out[:, 0, :]
        xi0_raw = init_out[:, 1, :]
        h0_raw = init_out[:, 2, :]
        a0_raw = init_out[:, 3, :]

        gate = torch.sigmoid(gate_logits)
        xi0 = torch.sigmoid(xi0_raw)
        h0 = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(h0_raw)
        a0 = torch.tanh(a0_raw)

        tf = fourier_features(t.reshape(B * N, 1), self.n_fourier).reshape(B, N, -1)
        emb_rep = emb.unsqueeze(1).expand(-1, N, -1)
        dyn_in = torch.cat([emb_rep, tf], dim=-1)
        dyn_out = self.dyn_net(dyn_in).view(B, N, 4, self.M)

        dxi_raw = dyn_out[:, :, 0, :]
        dh_raw = dyn_out[:, :, 1, :]
        alpha_raw = dyn_out[:, :, 2, :]
        vel_raw = dyn_out[:, :, 3, :]

        xi = torch.remainder(
            xi0.unsqueeze(1) + 0.35 * torch.tanh(dxi_raw) + self.vel_scale * t * torch.tanh(vel_raw),
            1.0,
        )
        h = torch.clamp(
            h0.unsqueeze(1) * (1.0 + 0.35 * torch.tanh(dh_raw)),
            min=self.h_min,
            max=self.h_max,
        )
        alpha = a0.unsqueeze(1) + 1.0 * torch.tanh(alpha_raw)
        gate_full = gate.unsqueeze(1).expand(-1, N, -1)
        return alpha, xi, h, gate_full

    def forward(self, p: torch.Tensor, xt: torch.Tensor):
        x = xt[:, :, 0:1]
        t = xt[:, :, 1:2]
        alpha, xi, h, gate = self.basis_params(p, t)
        d = periodic_distance_torch(x, xi)
        phi = torch.exp(-(d ** 2) / (2.0 * h ** 2))
        u = torch.sum((gate * alpha) * phi, dim=-1)
        return u, (gate, alpha, xi, h)


# ============================================================
# Single-instance dynamic-basis PINN (no meta-network)
# ============================================================
class SingleTaskAdvectionPINN(nn.Module):
    def __init__(
        self,
        M: int = 32,
        hidden_time: int = 96,
        n_fourier: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.25,
        vel_scale: float = 0.25,
    ) -> None:
        super().__init__()
        self.M = M
        self.n_fourier = n_fourier
        self.h_min = h_min
        self.h_max = h_max
        self.vel_scale = vel_scale

        self.gate_logits = nn.Parameter(torch.zeros(M))
        self.xi0_raw = nn.Parameter(torch.randn(M) * 0.25)
        self.h0_raw = nn.Parameter(torch.randn(M) * 0.10)
        self.a0_raw = nn.Parameter(torch.randn(M) * 0.10)

        in_dyn = 1 + 2 * n_fourier
        self.dyn_net = nn.Sequential(
            nn.Linear(in_dyn, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, hidden_time),
            nn.Tanh(),
            nn.Linear(hidden_time, 4 * M),
        )

    def basis_params(self, t: torch.Tensor):
        B, N, _ = t.shape
        gate = torch.sigmoid(self.gate_logits).view(1, 1, self.M).expand(B, N, -1)
        xi0 = torch.sigmoid(self.xi0_raw).view(1, 1, self.M).expand(B, N, -1)
        h0 = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(self.h0_raw).view(1, 1, self.M).expand(B, N, -1)
        a0 = torch.tanh(self.a0_raw).view(1, 1, self.M).expand(B, N, -1)

        tf = fourier_features(t.reshape(B * N, 1), self.n_fourier).reshape(B, N, -1)
        dyn_out = self.dyn_net(tf).view(B, N, 4, self.M)

        dxi_raw = dyn_out[:, :, 0, :]
        dh_raw = dyn_out[:, :, 1, :]
        alpha_raw = dyn_out[:, :, 2, :]
        vel_raw = dyn_out[:, :, 3, :]

        xi = torch.remainder(
            xi0 + 0.35 * torch.tanh(dxi_raw) + self.vel_scale * t * torch.tanh(vel_raw),
            1.0,
        )
        h = torch.clamp(
            h0 * (1.0 + 0.35 * torch.tanh(dh_raw)),
            min=self.h_min,
            max=self.h_max,
        )
        alpha = a0 + 1.0 * torch.tanh(alpha_raw)
        return alpha, xi, h, gate

    def forward(self, xt: torch.Tensor):
        x = xt[:, :, 0:1]
        t = xt[:, :, 1:2]
        alpha, xi, h, gate = self.basis_params(t)
        d = periodic_distance_torch(x, xi)
        phi = torch.exp(-(d ** 2) / (2.0 * h ** 2))
        u = torch.sum((gate * alpha) * phi, dim=-1)
        return u, (gate, alpha, xi, h)


# ============================================================
# Sampling
# ============================================================
def sample_p_batch(B: int, device: torch.device, x0_range=(X0_MIN, X0_MAX), nu_range=(NU_MIN, NU_MAX)) -> torch.Tensor:
    x0 = x0_range[0] + (x0_range[1] - x0_range[0]) * torch.rand(B, 1, device=device)
    u = torch.rand(B, 1, device=device)
    log_nu_min = math.log10(nu_range[0])
    log_nu_max = math.log10(nu_range[1])
    nu = 10.0 ** (log_nu_min + (log_nu_max - log_nu_min) * u)
    return torch.cat([x0, nu], dim=1)


def sample_interior_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_ic_points_batch(B: int, N: int, device: torch.device):
    x = torch.rand(B, N, 1, device=device)
    t = torch.zeros(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


def sample_periodic_points_batch(B: int, N: int, device: torch.device):
    t = torch.rand(B, N, 1, device=device)
    xL = torch.zeros(B, N, 1, device=device)
    xR = torch.ones(B, N, 1, device=device)
    return torch.cat([xL, t], dim=-1), torch.cat([xR, t], dim=-1)


def sample_near_ic_points_batch(B: int, N: int, device: torch.device, t_max: float = 0.15):
    x = torch.rand(B, N, 1, device=device)
    t = t_max * torch.rand(B, N, 1, device=device)
    return torch.cat([x, t], dim=-1)


# ============================================================
# Residuals / exact solution
# ============================================================
def advection_residual_batch_meta(model: nn.Module, p: torch.Tensor, xt_int: torch.Tensor) -> torch.Tensor:
    xt_int = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(p, xt_int)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]
    return grads[:, :, 1] + grads[:, :, 0]


def advection_residual_batch_single(model: nn.Module, xt_int: torch.Tensor) -> torch.Tensor:
    xt_int = xt_int.clone().detach().requires_grad_(True)
    u, _ = model(xt_int)
    grads = torch.autograd.grad(
        outputs=u,
        inputs=xt_int,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]
    return grads[:, :, 1] + grads[:, :, 0]


def exact_solution(x: np.ndarray, t: np.ndarray, x0: float, nu: float) -> np.ndarray:
    s = np.remainder(x - t, 1.0)
    return periodic_gaussian_numpy(s, x0, nu)


# ============================================================
# Training configs
# ============================================================
@dataclass
class MetaTrainConfig:
    B_pde: int = 4
    N_int: int = 256
    N_ic: int = 128
    N_per: int = 128
    N_near: int = 96
    epochs: int = 5000
    lr: float = 1e-3
    print_every: int = 100
    lambda_ic: float = 10.0
    lambda_per: float = 1.0
    lambda_center: float = 5e-4
    lambda_width: float = 2e-4


@dataclass
class SingleTrainConfig:
    B_pde: int = 1
    N_int: int = 256
    N_ic: int = 128
    N_per: int = 128
    N_near: int = 96
    epochs: int = 5000
    lr: float = 1e-3
    print_every: int = 100
    lambda_ic: float = 10.0
    lambda_per: float = 1.0
    lambda_center: float = 5e-4
    lambda_width: float = 2e-4


# ============================================================
# Training
# ============================================================
def train_meta_advection(model: nn.Module, device: torch.device, cfg: MetaTrainConfig):
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    best_loss = float("inf")
    best_state = None
    history = {k: [] for k in ["epoch", "loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]}
    t0 = time.perf_counter()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()

        if ep <= cfg.epochs // 2:
            nu_min_curr, nu_max_curr = NU_EASY, NU_MAX
        else:
            nu_min_curr, nu_max_curr = NU_MIN, NU_MAX

        p_batch = sample_p_batch(cfg.B_pde, device, nu_range=(nu_min_curr, nu_max_curr))
        xt_int = sample_interior_points_batch(cfg.B_pde, cfg.N_int, device)
        xt_ic = sample_ic_points_batch(cfg.B_pde, cfg.N_ic, device)
        xtL, xtR = sample_periodic_points_batch(cfg.B_pde, cfg.N_per, device)
        xt_near = sample_near_ic_points_batch(cfg.B_pde, cfg.N_near, device, t_max=0.15)

        res_int = advection_residual_batch_meta(model, p_batch, xt_int)
        res_near = advection_residual_batch_meta(model, p_batch, xt_near)
        loss_pde = torch.mean(res_int ** 2) + 0.5 * torch.mean(res_near ** 2)

        u_ic, _ = model(p_batch, xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_batch[:, 0:1].expand_as(x_ic)
        nu = p_batch[:, 1:2].expand_as(x_ic)
        u_ic_true = periodic_gaussian_torch(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        uL, _ = model(p_batch, xtL)
        uR, _ = model(p_batch, xtR)
        loss_per = torch.mean((uL - uR) ** 2)

        _, (gate_aux, alpha_aux, xi_aux, h_aux) = model(p_batch, xt_near)
        dxi_dt = xi_aux[:, 1:, :] - xi_aux[:, :-1, :] if xi_aux.shape[1] > 1 else xi_aux * 0.0
        loss_center = torch.mean(torch.abs(dxi_dt)) + 0.2 * torch.mean(torch.abs(gate_aux * alpha_aux))
        target_h = p_batch[:, 1:2].unsqueeze(1)
        loss_width = torch.mean((h_aux - torch.clamp(1.25 * target_h, model.h_min, model.h_max)) ** 2)

        loss = (
            loss_pde
            + cfg.lambda_ic * loss_ic
            + cfg.lambda_per * loss_per
            + cfg.lambda_center * loss_center
            + cfg.lambda_width * loss_width
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        history["epoch"].append(ep)
        history["loss"].append(float(loss.item()))
        history["loss_pde"].append(float(loss_pde.item()))
        history["loss_ic"].append(float(loss_ic.item()))
        history["loss_per"].append(float(loss_per.item()))
        history["loss_center"].append(float(loss_center.item()))
        history["loss_width"].append(float(loss_width.item()))

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % cfg.print_every == 0 or ep == 1:
            print(f"[KAPI] Epoch {ep:5d} | Loss {loss.item():.3e} | PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | PER {loss_per.item():.3e}")

    train_time = time.perf_counter() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, train_time


def train_single_advection(model: nn.Module, device: torch.device, cfg: SingleTrainConfig):
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    best_loss = float("inf")
    best_state = None
    history = {k: [] for k in ["epoch", "loss", "loss_pde", "loss_ic", "loss_per", "loss_center", "loss_width"]}
    p_x0 = torch.tensor([[X0_FIXED]], dtype=torch.float32, device=device)
    p_nu = torch.tensor([[NU_FIXED]], dtype=torch.float32, device=device)
    t0 = time.perf_counter()

    for ep in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad()

        xt_int = sample_interior_points_batch(cfg.B_pde, cfg.N_int, device)
        xt_ic = sample_ic_points_batch(cfg.B_pde, cfg.N_ic, device)
        xtL, xtR = sample_periodic_points_batch(cfg.B_pde, cfg.N_per, device)
        xt_near = sample_near_ic_points_batch(cfg.B_pde, cfg.N_near, device, t_max=0.15)

        res_int = advection_residual_batch_single(model, xt_int)
        res_near = advection_residual_batch_single(model, xt_near)
        loss_pde = torch.mean(res_int ** 2) + 0.5 * torch.mean(res_near ** 2)

        u_ic, _ = model(xt_ic)
        x_ic = xt_ic[:, :, 0]
        x0 = p_x0.expand_as(x_ic)
        nu = p_nu.expand_as(x_ic)
        u_ic_true = periodic_gaussian_torch(x_ic, x0, nu)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        uL, _ = model(xtL)
        uR, _ = model(xtR)
        loss_per = torch.mean((uL - uR) ** 2)

        _, (gate_aux, alpha_aux, xi_aux, h_aux) = model(xt_near)
        dxi_dt = xi_aux[:, 1:, :] - xi_aux[:, :-1, :] if xi_aux.shape[1] > 1 else xi_aux * 0.0
        loss_center = torch.mean(torch.abs(dxi_dt)) + 0.2 * torch.mean(torch.abs(gate_aux * alpha_aux))
        target_h = p_nu.unsqueeze(1)
        loss_width = torch.mean((h_aux - torch.clamp(1.25 * target_h, model.h_min, model.h_max)) ** 2)

        loss = (
            loss_pde
            + cfg.lambda_ic * loss_ic
            + cfg.lambda_per * loss_per
            + cfg.lambda_center * loss_center
            + cfg.lambda_width * loss_width
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        history["epoch"].append(ep)
        history["loss"].append(float(loss.item()))
        history["loss_pde"].append(float(loss_pde.item()))
        history["loss_ic"].append(float(loss_ic.item()))
        history["loss_per"].append(float(loss_per.item()))
        history["loss_center"].append(float(loss_center.item()))
        history["loss_width"].append(float(loss_width.item()))

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % cfg.print_every == 0 or ep == 1:
            print(f"[PINN] Epoch {ep:5d} | Loss {loss.item():.3e} | PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | PER {loss_per.item():.3e}")

    train_time = time.perf_counter() - t0
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, train_time


# ============================================================
# Evaluation
# ============================================================
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def evaluate_meta(model: DynamicMetaSPINN, device: torch.device, Nx: int = 200, Nt: int = 200) -> Dict[str, np.ndarray]:
    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)
    p_test = torch.tensor([[X0_FIXED, NU_FIXED]], dtype=torch.float32, device=device)

    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        u_pred_flat, _ = model(p_test, xt_grid_t)
    infer_time = time.perf_counter() - t0

    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)
    u_exact = exact_solution(X, T, X0_FIXED, NU_FIXED)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    t_vals = np.linspace(0.0, 1.0, 60)
    t_geom = torch.tensor(t_vals, dtype=torch.float32, device=device).view(1, 60, 1)
    with torch.no_grad():
        alpha_g, xi_g, h_g, gate_g = model.basis_params(p_test, t_geom)

    return dict(
        x=x, t=t, u_exact=u_exact, u_pred=u_pred, abs_error=abs_err,
        rel_l2=rel_l2, infer_time=infer_time,
        xi=xi_g[0].cpu().numpy(), h=h_g[0].cpu().numpy(),
        alpha=alpha_g[0].cpu().numpy(), gate=gate_g[0].cpu().numpy(),
        t_geom=t_vals
    )


def evaluate_single(model: SingleTaskAdvectionPINN, device: torch.device, Nx: int = 200, Nt: int = 200) -> Dict[str, np.ndarray]:
    x = np.linspace(0.0, 1.0, Nx, endpoint=False)
    t = np.linspace(0.0, 1.0, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    xt_grid = np.stack([X.ravel(), T.ravel()], axis=1)
    xt_grid_t = torch.tensor(xt_grid, dtype=torch.float32, device=device).view(1, -1, 2)

    model.eval()
    t0 = time.perf_counter()
    with torch.no_grad():
        u_pred_flat, _ = model(xt_grid_t)
    infer_time = time.perf_counter() - t0

    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)
    u_exact = exact_solution(X, T, X0_FIXED, NU_FIXED)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    t_vals = np.linspace(0.0, 1.0, 60)
    t_geom = torch.tensor(t_vals, dtype=torch.float32, device=device).view(1, 60, 1)
    with torch.no_grad():
        alpha_g, xi_g, h_g, gate_g = model.basis_params(t_geom)

    return dict(
        x=x, t=t, u_exact=u_exact, u_pred=u_pred, abs_error=abs_err,
        rel_l2=rel_l2, infer_time=infer_time,
        xi=xi_g[0].cpu().numpy(), h=h_g[0].cpu().numpy(),
        alpha=alpha_g[0].cpu().numpy(), gate=gate_g[0].cpu().numpy(),
        t_geom=t_vals
    )


# ============================================================
# Saving / plotting
# ============================================================
def save_summary(out_dir: Path, kapi_eval, pinn_eval, kapi_train_time, pinn_train_time, kapi_params, pinn_params):
    rows = [
        {
            "method": "KAPI_meta_solver",
            "rel_l2_error": float(kapi_eval["rel_l2"]),
            "offline_train_time_sec": float(kapi_train_time),
            "test_time_inference_sec": float(kapi_eval["infer_time"]),
            "trainable_params": int(kapi_params),
            "requires_retraining_per_new_task": "No",
        },
        {
            "method": "Single_instance_PINN",
            "rel_l2_error": float(pinn_eval["rel_l2"]),
            "offline_train_time_sec": float(pinn_train_time),
            "test_time_inference_sec": float(pinn_eval["infer_time"]),
            "trainable_params": int(pinn_params),
            "requires_retraining_per_new_task": "Yes",
        },
    ]

    with open(out_dir / "ablation_summary.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    payload = {
        "fixed_task": {"x0": X0_FIXED, "nu": NU_FIXED},
        "kapi": rows[0],
        "single_instance_pinn": rows[1],
    }
    with open(out_dir / "ablation_summary.json", "w") as f:
        json.dump(payload, f, indent=2)

    np.savez_compressed(
        out_dir / "ablation_outputs.npz",
        x=kapi_eval["x"].astype(np.float32),
        t=kapi_eval["t"].astype(np.float32),
        u_exact=kapi_eval["u_exact"].astype(np.float32),
        u_kapi=kapi_eval["u_pred"].astype(np.float32),
        e_kapi=kapi_eval["abs_error"].astype(np.float32),
        u_pinn=pinn_eval["u_pred"].astype(np.float32),
        e_pinn=pinn_eval["abs_error"].astype(np.float32),
        kapi_rel_l2=np.float32(kapi_eval["rel_l2"]),
        pinn_rel_l2=np.float32(pinn_eval["rel_l2"]),
        kapi_train_time=np.float32(kapi_train_time),
        pinn_train_time=np.float32(pinn_train_time),
    )


def make_plots(out_dir: Path, kapi_history, pinn_history, kapi_eval, pinn_eval):
    plt.figure(figsize=(7.4, 4.8))
    plt.plot(kapi_history["epoch"], kapi_history["loss"], lw=2, label="KAPI total")
    plt.plot(pinn_history["epoch"], pinn_history["loss"], lw=2, label="Single-instance PINN total")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Advection ablation: training curves")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "ablation_training_curves.png", dpi=300)
    plt.close()

    u_exact = kapi_eval["u_exact"]
    u_kapi = kapi_eval["u_pred"]
    e_kapi = kapi_eval["abs_error"]
    u_pinn = pinn_eval["u_pred"]
    e_pinn = pinn_eval["abs_error"]

    sol_vmin = min(np.min(u_exact), np.min(u_kapi), np.min(u_pinn))
    sol_vmax = max(np.max(u_exact), np.max(u_kapi), np.max(u_pinn))
    err_vmax = max(np.max(e_kapi), np.max(e_pinn))

    fig, axs = plt.subplots(2, 3, figsize=(12.8, 7.2), constrained_layout=True)
    im0 = axs[0, 0].imshow(u_exact.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
    axs[0, 0].set_title("Exact")
    im1 = axs[0, 1].imshow(u_kapi.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
    axs[0, 1].set_title("KAPI")
    im2 = axs[0, 2].imshow(u_pinn.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
    axs[0, 2].set_title("Single-instance PINN")

    axs[1, 0].axis("off")
    im3 = axs[1, 1].imshow(e_kapi.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0.0, vmax=err_vmax)
    axs[1, 1].set_title(r"$|u_{KAPI}-u_{exact}|$")
    im4 = axs[1, 2].imshow(e_pinn.T, origin="lower", extent=[0, 1, 0, 1], aspect="auto", vmin=0.0, vmax=err_vmax)
    axs[1, 2].set_title(r"$|u_{PINN}-u_{exact}|$")

    for ax in [axs[0, 0], axs[0, 1], axs[0, 2], axs[1, 1], axs[1, 2]]:
        ax.set_xlabel("x")
        ax.set_ylabel("t")

    cbar1 = fig.colorbar(im2, ax=axs[0, :], shrink=0.9)
    cbar1.set_label(r"$u(x,t)$")
    cbar2 = fig.colorbar(im4, ax=axs[1, 1:], shrink=0.9)
    cbar2.set_label("Absolute Error")
    fig.suptitle(rf"Advection ablation on fixed task $(x_0,\nu)=({X0_FIXED:.2f},{NU_FIXED:.2f})$")
    fig.savefig(out_dir / "ablation_comparison.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    fig, axs = plt.subplots(1, 2, figsize=(10.8, 4.8), constrained_layout=True)
    sc = None
    for ax, res, ttl in zip(axs, [kapi_eval, pinn_eval], ["KAPI geometry", "Single-instance PINN geometry"]):
        xi = res["xi"]
        h = res["h"]
        alpha = res["alpha"]
        gate = res["gate"]
        tvals = res["t_geom"]
        weights = np.abs(alpha * gate)

        X_pts = xi.reshape(-1)
        T_pts = np.repeat(tvals, xi.shape[1])
        S_pts = np.clip(120.0 * h.reshape(-1), 8.0, 80.0)
        C_pts = weights.reshape(-1)

        sc = ax.scatter(X_pts, T_pts, c=C_pts, s=S_pts, cmap="viridis", alpha=0.85)
        ax.scatter([X0_FIXED], [0.0], color="red", marker="x", s=90, label="IC center")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(ttl)
        ax.legend(loc="upper right")

    cbar = fig.colorbar(sc, ax=axs, shrink=0.9)
    cbar.set_label(r"$|\alpha_j g_j|$")
    fig.savefig(out_dir / "ablation_geometry.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


# ============================================================
# Main
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="Advection ablation: KAPI vs single-instance PINN")
    parser.add_argument("--gpu", action="store_true", help="Use GPU if available")
    parser.add_argument("--outdir", type=str, default="advection_ablation_run")
    parser.add_argument("--epochs_kapi", type=int, default=5000)
    parser.add_argument("--epochs_pinn", type=int, default=5000)
    parser.add_argument("--lr_kapi", type=float, default=1e-3)
    parser.add_argument("--lr_pinn", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=32)
    parser.add_argument("--B_pde", type=int, default=4)
    parser.add_argument("--N_int", type=int, default=256)
    parser.add_argument("--N_ic", type=int, default=128)
    parser.add_argument("--N_per", type=int, default=128)
    parser.add_argument("--N_near", type=int, default=96)
    parser.add_argument("--eval_Nx", type=int, default=200)
    parser.add_argument("--eval_Nt", type=int, default=200)
    args, _unknown = parser.parse_known_args()

    device = get_device(args.gpu)
    out_dir = Path(args.outdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print("Using device:", device)
    print(f"Running advection ablation on fixed task (x0, nu)=({X0_FIXED}, {NU_FIXED})")

    meta_cfg = MetaTrainConfig(
        B_pde=args.B_pde,
        N_int=args.N_int,
        N_ic=args.N_ic,
        N_per=args.N_per,
        N_near=args.N_near,
        epochs=args.epochs_kapi,
        lr=args.lr_kapi,
    )
    single_cfg = SingleTrainConfig(
        B_pde=1,
        N_int=args.N_int,
        N_ic=args.N_ic,
        N_per=args.N_per,
        N_near=args.N_near,
        epochs=args.epochs_pinn,
        lr=args.lr_pinn,
    )

    kapi_model = DynamicMetaSPINN(M=args.M).to(device)
    kapi_model, kapi_history, kapi_train_time = train_meta_advection(kapi_model, device, meta_cfg)

    pinn_model = SingleTaskAdvectionPINN(M=args.M).to(device)
    pinn_model, pinn_history, pinn_train_time = train_single_advection(pinn_model, device, single_cfg)

    torch.save(kapi_model.state_dict(), out_dir / "kapi_advection_model.pt")
    torch.save(pinn_model.state_dict(), out_dir / "single_instance_advection_pinn_model.pt")

    kapi_eval = evaluate_meta(kapi_model, device=device, Nx=args.eval_Nx, Nt=args.eval_Nt)
    pinn_eval = evaluate_single(pinn_model, device=device, Nx=args.eval_Nx, Nt=args.eval_Nt)

    kapi_params = count_params(kapi_model)
    pinn_params = count_params(pinn_model)

    save_summary(out_dir, kapi_eval, pinn_eval, kapi_train_time, pinn_train_time, kapi_params, pinn_params)
    make_plots(out_dir, kapi_history, pinn_history, kapi_eval, pinn_eval)

    print("\nSummary:")
    print(f"  KAPI relL2       : {kapi_eval['rel_l2']:.3e}")
    print(f"  PINN relL2       : {pinn_eval['rel_l2']:.3e}")
    print(f"  KAPI train time  : {kapi_train_time:.2f} s")
    print(f"  PINN train time  : {pinn_train_time:.2f} s")
    print(f"  KAPI inference   : {kapi_eval['infer_time']:.6f} s")
    print(f"  PINN inference   : {pinn_eval['infer_time']:.6f} s")
    print(f"  KAPI params      : {kapi_params}")
    print(f"  PINN params      : {pinn_params}")
    print(f"\nSaved outputs in: {out_dir}")


if __name__ == "__main__":
    main()

Using device: cpu
Running advection ablation on fixed task (x0, nu)=(0.5, 0.07)
[KAPI] Epoch     1 | Loss 6.045e+00 | PDE 4.232e+00 | IC 1.813e-01 | PER 2.125e-18
[KAPI] Epoch   100 | Loss 4.999e-01 | PDE 2.150e-01 | IC 2.849e-02 | PER 6.103e-17
[KAPI] Epoch   200 | Loss 2.691e-01 | PDE 1.733e-01 | IC 9.573e-03 | PER 3.135e-16
[KAPI] Epoch   300 | Loss 1.285e-01 | PDE 1.014e-01 | IC 2.702e-03 | PER 6.324e-16
[KAPI] Epoch   400 | Loss 1.124e-01 | PDE 8.960e-02 | IC 2.282e-03 | PER 5.696e-16
[KAPI] Epoch   500 | Loss 8.354e-02 | PDE 6.591e-02 | IC 1.761e-03 | PER 1.017e-15
[KAPI] Epoch   600 | Loss 5.594e-02 | PDE 4.204e-02 | IC 1.387e-03 | PER 1.026e-15
[KAPI] Epoch   700 | Loss 4.860e-02 | PDE 2.860e-02 | IC 1.998e-03 | PER 9.860e-16
[KAPI] Epoch   800 | Loss 8.857e-02 | PDE 7.985e-02 | IC 8.691e-04 | PER 3.105e-15
[KAPI] Epoch   900 | Loss 4.522e-02 | PDE 4.225e-02 | IC 2.952e-04 | PER 3.384e-15
[KAPI] Epoch  1000 | Loss 1.607e-02 | PDE 1.159e-02 | IC 4.452e-04 | PER 1.914e-15
[KAPI] 